# Recovery against the horizon

Sample paths of the ARI against $n$, at fixed $N$ — the published Figure 4, with the outer
level of randomness the design was missing.

That figure drew the kernels once and replicated the data 50 times, so everything it showed
was a property of one draw of a $K = 4$ mixture at $\alpha = 0.3$. The rise from 0.67 to 1
it reported may or may not be typical; nothing in the experiment separated the mixture from
the draw. Here several mixtures are drawn, each held fixed while its datasets are
replicated, so the figure carries both: the spread between replicates of one mixture, and
the spread between mixtures.

Everything else is unchanged. A replicate is a genuine sample path — the same $N$ sequences
read at growing lengths, through the nested prefixes of `om_matrices` — so the thin lines
show the variability of a single experiment, not an interval around the mean. The three
estimators share these axes exactly, so the figures stack and compare.

Exact recovery is the event $\mathrm{ARI} = 1$ and is read off the top of the plot; its
probability is tabulated under each figure.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments import wilson_interval
from figures import PAPER_STYLE

## Setup

`sweep_path.py` writes the tidy table; the figures read it. Recomputing takes about twenty
minutes at $N = 200$, and roughly sixteen times that at the published $N = 800$.

In [ ]:
RECOMPUTE = False

RESULTS = Path("results")
FIGURES = Path("Figures/Recovery")
FIGURES.mkdir(parents=True, exist_ok=True)

if RECOMPUTE:
    import subprocess
    subprocess.run(["python3", "sweep_path.py", "--tag", "main"], check=True)

path = pd.read_csv(RESULTS / "path_cluster_main.csv")
eta  = pd.read_csv(RESULTS / "path_eta_main.csv")

ALPHA_FIX = float(path.alpha.iloc[0])
K_FIX     = int(path.K.iloc[0])
PATH_N    = int(path.N.iloc[0])
HORIZONS  = np.sort(path.n.unique())
N_MIX     = path.mixture_id.nunique()
N_DATA    = path.dataset_id.nunique()

print(f"alpha = {ALPHA_FIX}, K = {K_FIX}, N = {PATH_N}; "
      f"{N_MIX} mixtures x {N_DATA} datasets over horizons {list(HORIZONS)}")

## The figure

Same layout as `plot_gamma_convergence` in `om_convergence.ipynb`: one thin line per
replicate, the mean over them on top. The six intermediate lines are the per-mixture means —
the level the published figure could not show, since it had a single mixture. Where they
fan out, the horizon needed for recovery depends on which kernels were drawn; where they
collapse onto the mean, it does not.

In [ ]:
COL = {"single":  "#1b6ca8",       # the blue of om_convergence.ipynb
       "average": "#6a51a3",       # a third hue, so no two estimators share a colour
       "pam":     "#c0392b"}
NAMES = {"single": "single linkage", "average": "average linkage", "pam": "PAM"}


def curves(df, algorithm, value="ari"):
    """(replicate, horizon) array, plus the per-mixture means."""
    sub = df[df.algorithm == algorithm]
    wide = sub.pivot_table(index=["mixture_id", "dataset_id"], columns="n", values=value)
    wide = wide.reindex(columns=HORIZONS)
    per_mixture = sub.pivot_table(index="mixture_id", columns="n", values=value)
    return wide.to_numpy(), per_mixture.reindex(columns=HORIZONS).to_numpy()


def plot_ari_curve(algorithm, filename=None):
    reps, per_mixture = curves(path, algorithm)
    colour = COL[algorithm]
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        ax.plot(HORIZONS, reps.T, color=colour, lw=0.6, alpha=0.13)
        ax.plot(HORIZONS, per_mixture.T, color=colour, lw=0.9, alpha=0.5)
        ax.plot(HORIZONS, reps.mean(0), color=colour, lw=1.9,
                label=rf"mean over {reps.shape[0]} replicates")
        ax.plot([], [], color=colour, lw=0.9, alpha=0.5,
                label=rf"{per_mixture.shape[0]} mixture means")
        ax.set_xlabel(r"$n$, length of a sequence")
        ax.set_ylabel(f"ARI, {NAMES[algorithm]}")
        ax.set_ylim(-0.04, 1.04)
        ax.axhline(0.5, color="0.6", lw=0.7, ls=(0, (1, 3)), zorder=0)
        ax.set_title(rf"{NAMES[algorithm]} --- $N = {PATH_N}$, $K = {K_FIX}$, "
                     rf"$\alpha = {ALPHA_FIX}$", fontsize=10)
        ax.legend(loc="lower right", handlelength=2.6, borderaxespad=0.6)
        fig.tight_layout()
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(FIGURES / f"{filename}.{ext}", bbox_inches="tight")
            print("figure written to", FIGURES / f"{filename}.pdf")
        plt.show()
    return fig


def exact_recovery_table(algorithm):
    """P(ARI = 1) at each horizon, with its Wilson interval."""
    sub = path[path.algorithm == algorithm]
    rows = []
    for n in HORIZONS:
        s = sub[sub.n == n].exact_recovery
        lo, hi = wilson_interval(int(s.sum()), len(s))
        rows.append({"n": int(n), "P(exact)": f"{s.mean():.2f}",
                     "95% CI": f"[{lo:.2f}, {hi:.2f}]",
                     "mean ARI": f"{sub[sub.n == n].ari.mean():.3f}"})
    return pd.DataFrame(rows)

### Average linkage

In [ ]:
_ = plot_ari_curve("average", filename="ari_path_average_linkage")
print(exact_recovery_table("average").to_string(index=False))

### PAM

In [ ]:
_ = plot_ari_curve("pam", filename="ari_path_pam")
print(exact_recovery_table("pam").to_string(index=False))

## What the mixtures do

The per-mixture means above fan out; the table says by how much. A horizon at which one
mixture is recovered every time can leave another below a half, which is the reason for
drawing more than one.

$\eta_n$ is estimated on an independent sample at the same horizons, so the ordering of the
mixtures can be read against the geometry rather than against their labels.

In [ ]:
rows = []
for m in sorted(path.mixture_id.unique()):
    row = {"mixture": m}
    e = eta[eta.mixture_id == m]
    row["eta at n=%d" % HORIZONS[-1]] = f"{e[e.n == HORIZONS[-1]].eta_hat.iloc[0]:+.3f}"
    row["verdict"] = e[e.n == HORIZONS[-1]].separation_status.iloc[0]
    for algo in ("average", "pam"):
        s = path[(path.mixture_id == m) & (path.algorithm == algo)
                 & (path.n == HORIZONS[-1])]
        row[f"{algo}: P(exact)"] = f"{s.exact_recovery.mean():.2f}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## Appendix: single linkage

The estimator Theorem 3.5 is about, on the same axes, where the published Figure 9 had it.

In [ ]:
_ = plot_ari_curve("single", filename="ari_path_single_linkage")
print(exact_recovery_table("single").to_string(index=False))

certified = path[path.algorithm == "pam"]
print(f"\nPAM certified one-swap stationary in "
      f"{int(certified.pam_one_swap_certified.sum())}/{len(certified)} runs; "
      f"hit the swap cap {int(certified.pam_hit_cap.sum())} times")